# 04. Named Entity Recognition (NER)

**Objective:** Extract and analyze companies, tickers, executives, locations, and organizations from financial news

**Methods:**
- spaCy NER (en_core_web_trf for transformer-based accuracy)
- BERT-NER (dslim/bert-base-NER for domain-specific entities)
- Custom financial entity patterns
- Entity co-mention network analysis

In [ ]:


import pandas as pd
import numpy as np
import json
from pathlib import Path
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import warnings
import os

# Suppress warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Import torch FIRST
import torch
print(f"\n  PyTorch {torch.__version__} loaded")

# Import transformers (PyTorch backend only - TensorFlow removed)
from transformers import pipeline
print(f"  Transformers loaded")

# Import spaCy  
import spacy
print(f"  spaCy {spacy.__version__} loaded")

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Network analysis
import networkx as nx

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print(f"\nDevice: {'GPU (CUDA)' if torch.cuda.is_available() else 'CPU'}")
print(f"\n  All libraries loaded successfully!")

  NAMED ENTITY RECOGNITION - PyTorch 2.5.1 (DLL Fixed!)

  PyTorch 2.5.1+cpu loaded successfully!
  Transformers loaded successfully!
  spaCy 3.7.2 loaded successfully!

Device: CPU

  All libraries loaded successfully!


## 1. Setup Paths and Configuration

In [2]:
# Setup paths
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / '01_DATA'
FEATURES_DIR = DATA_DIR / 'features'
RESULTS_DIR = BASE_DIR / '03_RESULTS'
OUTPUTS_DIR = RESULTS_DIR / 'outputs'
VIZ_DIR = RESULTS_DIR / 'visualizations'
CONFIG_DIR = BASE_DIR / '06_CONFIG'

# Create directories
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
VIZ_DIR.mkdir(parents=True, exist_ok=True)

# Load configuration
with open(CONFIG_DIR / 'framework_config.json', 'r') as f:
    config = json.load(f)

with open(CONFIG_DIR / 'financial_lexicon.json', 'r') as f:
    lexicon = json.load(f)

print(f" Base directory: {BASE_DIR}")
print(f" Features directory: {FEATURES_DIR}")
print(f" Results directory: {RESULTS_DIR}")
print(f" Configuration loaded")

 Base directory: c:\Users\HARSHIT\Desktop\p\nlp analysis
 Features directory: c:\Users\HARSHIT\Desktop\p\nlp analysis\01_DATA\features
 Results directory: c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS
 Configuration loaded


## 2. Load Sentiment-Enriched Dataset

In [3]:
# Load dataset with sentiment from notebook 03
df = pd.read_csv(FEATURES_DIR / 'dataset_with_sentiment.csv')
df['date'] = pd.to_datetime(df['date'])

print(f" Loaded {len(df):,} articles")
print(f" Date range: {df['date'].min()} to {df['date'].max()}")
print(f" Companies: {df['ticker'].nunique()}")
print(f"\n Sentiment columns available: {[col for col in df.columns if 'sentiment' in col.lower() or 'vader' in col.lower() or 'textblob' in col.lower()]}")

df.head()

 Loaded 63 articles
 Date range: 2025-09-26 07:00:00 to 2026-01-18 12:19:48
 Companies: 19

 Sentiment columns available: ['finbert_sentiment', 'vader_compound', 'vader_label', 'textblob_polarity', 'textblob_label']


,date,title,text,source,url,ticker,publisher,query,article_length,word_count,collection_date,finbert_label,finbert_score,finbert_sentiment,vader_compound,vader_label,textblob_polarity,textblob_label,date_only
0,2026-01-18 12:19:48,American Express Company $AXP Shares Acquired ...,American Express Company $AXP Shares Acquired ...,Google News,https://news.google.com/rss/articles/CBMi2wFBV...,AXP,NaN,AXP stock news,106,13,2026-01-18 18:10:41.105671,positive,0.2960,0.2960,0.2960,positive,0.000000,neutral,2026-01-18
1,2026-01-18 12:15:12,Apple stock price slips to $255 ahead of a hol...,Apple stock price slips to $255 ahead of a hol...,Google News,https://news.google.com/rss/articles/CBMiuAFBV...,AAPL,NaN,AAPL stock news,121,18,2026-01-18 18:10:41.105671,neutral,0.0000,0.0000,0.0000,neutral,0.000000,neutral,2026-01-18
2,2026-01-18 12:00:26,Amgen Inc. (NASDAQ:AMGN) is largely controlled...,Amgen Inc. (NASDAQ:AMGN) is largely controlled...,Google News,https://news.google.com/rss/articles/CBMigAFBV...,AMGN,NaN,AMGN stock news,128,16,2026-01-18 18:10:41.105671,neutral,0.0000,0.0000,0.0000,neutral,0.407143,positive,2026-01-18
3,2026-01-18 11:43:58,BAC Stock Gains On Bullish Q4 Print Guides For...,BAC Stock Gains On Bullish Q4 Print Guides For...,Google News,https://news.google.com/rss/articles/CBMisgFBV...,BAC,NaN,BAC stock news,100,15,2026-01-18 18:10:41.105671,positive,0.6124,0.6124,0.6124,positive,0.000000,neutral,2026-01-18
4,2026-01-18 11:26:14,Strong Analyst Confidence in Alibaba Group Hol...,Strong Analyst Confidence in Alibaba Group Hol...,Google News,https://news.google.com/rss/articles/CBMiswFBV...,BABA,NaN,BABA stock news,102,13,2026-01-18 18:10:41.105671,positive,0.7650,0.7650,0.7650,positive,0.433333,positive,2026-01-18


## 3. Load spaCy NER Model

In [4]:


try:
    nlp = spacy.load("en_core_web_trf")
    print(" Loaded en_core_web_trf (transformer model)")
except:
    try:
        nlp = spacy.load("en_core_web_sm")
        print(" Using en_core_web_sm (smaller model - consider installing en_core_web_trf)")
    except:
        print(" No spaCy model found. Installing en_core_web_sm...")
        import subprocess
        subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"], check=True)
        nlp = spacy.load("en_core_web_sm")
        print(" Installed and loaded en_core_web_sm")

# Test NER
test_text = "Apple CEO Tim Cook announced record iPhone sales, while Microsoft and Amazon compete in cloud services."
doc = nlp(test_text)
print(f"\n Test entities found: {[(ent.text, ent.label_) for ent in doc.ents]}")

 Using en_core_web_sm (smaller model - consider installing en_core_web_trf)

 Test entities found: [('Apple', 'ORG'), ('Tim Cook', 'PERSON'), ('Microsoft', 'ORG'), ('Amazon', 'ORG')]


## 4. Load BERT-NER Model

In [6]:
# Load BERT-NER model for additional entity extraction
device = 0 if torch.cuda.is_available() else -1
print(f" Using device: {'GPU (CUDA)' if device == 0 else 'CPU'}")

bert_ner = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple",
    device=device
)

# Test BERT-NER
bert_results = bert_ner(test_text)
entities_str = [(r['word'], r['entity_group'], f"{r['score']:.2f}") for r in bert_results]
print(f"\n BERT-NER entities: {entities_str}")

 Using device: CPU


Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).



 BERT-NER entities: [('Apple', 'ORG', '1.00'), ('Tim Cook', 'PER', '1.00'), ('iPhone', 'MISC', '1.00'), ('Microsoft', 'ORG', '1.00'), ('Amazon', 'ORG', '1.00')]


## 5. Extract Entities with spaCy

In [7]:
def extract_spacy_entities(text, nlp_model):
    """Extract entities using spaCy NER"""
    if pd.isna(text) or not isinstance(text, str):
        return []
    
    # Process with spaCy (truncate to 1M chars to avoid memory issues)
    doc = nlp_model(text[:1000000])
    
    entities = []
    for ent in doc.ents:
        entities.append({
            'text': ent.text,
            'label': ent.label_,
            'start': ent.start_char,
            'end': ent.end_char
        })
    
    return entities

print(" Extracting entities with spaCy...")

# Process in batches for efficiency
spacy_entities_list = []
batch_size = 100

for i in tqdm(range(0, len(df), batch_size), desc="spaCy NER"):
    batch = df.iloc[i:i+batch_size]['text'].tolist()
    
    # Use nlp.pipe for efficient batch processing
    docs = list(nlp.pipe(batch, disable=['lemmatizer', 'textcat']))
    
    for doc in docs:
        entities = []
        for ent in doc.ents:
            entities.append({
                'text': ent.text,
                'label': ent.label_,
                'start': ent.start_char,
                'end': ent.end_char
            })
        spacy_entities_list.append(entities)

df['spacy_entities'] = spacy_entities_list

# Count entity types
spacy_entity_types = Counter()
for entities in df['spacy_entities']:
    for ent in entities:
        spacy_entity_types[ent['label']] += 1

print(f"\n Extracted {sum(len(e) for e in spacy_entities_list):,} entities")
print(f"\n Top entity types (spaCy):")
for label, count in spacy_entity_types.most_common(10):
    print(f"  {label}: {count:,}")

 Extracting entities with spaCy...


spaCy NER:   0%|          | 0/1 [00:00<?, ?it/s]


 Extracted 175 entities

 Top entity types (spaCy):
  ORG: 110
  DATE: 17
  MONEY: 13
  PERSON: 10
  GPE: 7
  PERCENT: 6
  NORP: 3
  WORK_OF_ART: 3
  CARDINAL: 3
  PRODUCT: 1


## 6. Extract Entities with BERT-NER

In [8]:
def extract_bert_entities(text, bert_model):
    """Extract entities using BERT-NER"""
    if pd.isna(text) or not isinstance(text, str):
        return []
    
    # BERT has token limit of 512
    text = text[:2000]  # Roughly 512 tokens
    
    try:
        results = bert_model(text)
        entities = []
        for r in results:
            entities.append({
                'text': r['word'].replace('##', ''),  # Remove BERT subword marker
                'label': r['entity_group'],
                'score': r['score'],
                'start': r['start'],
                'end': r['end']
            })
        return entities
    except:
        return []

print(" Extracting entities with BERT-NER...")

bert_entities_list = []
batch_size = 32 if device == 0 else 8

for i in tqdm(range(0, len(df), batch_size), desc="BERT-NER"):
    batch = df.iloc[i:i+batch_size]['text'].tolist()
    
    for text in batch:
        entities = extract_bert_entities(text, bert_ner)
        bert_entities_list.append(entities)

df['bert_entities'] = bert_entities_list

# Count entity types
bert_entity_types = Counter()
for entities in df['bert_entities']:
    for ent in entities:
        bert_entity_types[ent['label']] += 1

print(f"\n Extracted {sum(len(e) for e in bert_entities_list):,} entities")
print(f"\n Top entity types (BERT-NER):")
for label, count in bert_entity_types.most_common(10):
    print(f"  {label}: {count:,}")

 Extracting entities with BERT-NER...


BERT-NER:   0%|          | 0/8 [00:00<?, ?it/s]


 Extracted 199 entities

 Top entity types (BERT-NER):
  ORG: 168
  MISC: 21
  LOC: 7
  PER: 3


## 7. Build Financial Entity Database

In [9]:
# Combine entities from both models
def combine_entities(spacy_ents, bert_ents):
    """Combine and deduplicate entities from both models"""
    all_entities = []
    
    # Add spaCy entities
    for ent in spacy_ents:
        all_entities.append({
            'text': ent['text'],
            'label': ent['label'],
            'source': 'spacy'
        })
    
    # Add BERT entities
    for ent in bert_ents:
        all_entities.append({
            'text': ent['text'],
            'label': ent['label'],
            'source': 'bert'
        })
    
    return all_entities

df['all_entities'] = df.apply(
    lambda row: combine_entities(row['spacy_entities'], row['bert_entities']),
    axis=1
)

# Build entity database
entity_database = defaultdict(lambda: {'count': 0, 'articles': set(), 'types': Counter(), 'tickers': set()})

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Building entity database"):
    for ent in row['all_entities']:
        text = ent['text'].strip()
        if len(text) < 2:  # Skip single characters
            continue
        
        entity_database[text]['count'] += 1
        entity_database[text]['articles'].add(idx)
        entity_database[text]['types'][ent['label']] += 1
        if pd.notna(row['ticker']):
            entity_database[text]['tickers'].add(row['ticker'])

# Convert to DataFrame
entity_records = []
for entity_text, data in entity_database.items():
    entity_records.append({
        'entity': entity_text,
        'count': data['count'],
        'num_articles': len(data['articles']),
        'primary_type': data['types'].most_common(1)[0][0] if data['types'] else 'UNKNOWN',
        'all_types': ','.join([f"{k}:{v}" for k, v in data['types'].most_common()]),
        'tickers': ','.join(sorted(data['tickers'])),
        'num_tickers': len(data['tickers'])
    })

entity_df = pd.DataFrame(entity_records).sort_values('count', ascending=False)

print(f"\n Entity Database Statistics:")
print(f"  Total unique entities: {len(entity_df):,}")
print(f"  Entities mentioned 10+ times: {(entity_df['count'] >= 10).sum():,}")
print(f"  Entities mentioned 100+ times: {(entity_df['count'] >= 100).sum():,}")

# Save entity database
entity_df.to_csv(OUTPUTS_DIR / 'financial_entity_database.csv', index=False)
print(f"\n Saved entity database to {OUTPUTS_DIR / 'financial_entity_database.csv'}")

entity_df.head(20)

Building entity database:   0%|          | 0/63 [00:00<?, ?it/s]


 Entity Database Statistics:
  Total unique entities: 226
  Entities mentioned 10+ times: 1
  Entities mentioned 100+ times: 0

 Saved entity database to c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\financial_entity_database.csv


,entity,count,num_articles,primary_type,all_types,tickers,num_tickers
16,Yahoo Finance,12,12,ORG,ORG:12,"ABT,ACN,AMD,AMGN,APA,AVB,AZN,BABA,BBD",9
90,Boeing,7,4,ORG,ORG:7,BA,1
23,Abbott Laboratories,7,4,ORG,ORG:7,ABT,1
50,2026,6,5,DATE,DATE:6,"AAPL,AMD,AVGO,BA",4
137,Bank of America,6,3,ORG,ORG:6,BAC,1
89,BA,6,4,ORG,"ORG:5,NORP:1","BA,BABA",2
25,ABT,5,4,ORG,ORG:5,ABT,1
30,AI,5,4,ORG,"ORG:3,MISC:2","ADBE,AMD,AMT",3
72,APA,5,3,ORG,ORG:5,APA,1
116,Titan,5,4,GPE,"GPE:4,ORG:1","ACN,AMT,AVB",3


## 8. Entity Type Analysis

In [10]:
# Count entities by type
entity_type_counts = entity_df['primary_type'].value_counts()

# Create visualization
fig = px.bar(
    x=entity_type_counts.index,
    y=entity_type_counts.values,
    title="Entity Types Distribution",
    labels={'x': 'Entity Type', 'y': 'Count'},
    color=entity_type_counts.values,
    color_continuous_scale='Viridis'
)

fig.update_layout(
    height=500,
    showlegend=False,
    xaxis_tickangle=-45
)

fig.write_html(VIZ_DIR / 'entity_type_distribution.html')
fig.show()

# Save entity type counts
entity_type_counts.to_csv(OUTPUTS_DIR / 'entity_type_counts.csv')
print(f"\n Saved entity type counts to {OUTPUTS_DIR / 'entity_type_counts.csv'}")


 Saved entity type counts to c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\entity_type_counts.csv


## 9. Top Entities by Category

In [11]:
# Get top entities for each category
categories = ['ORG', 'PERSON', 'GPE', 'PRODUCT', 'LOC']
top_n = 20

for category in categories:
    top_entities = entity_df[entity_df['primary_type'] == category].head(top_n)
    
    if len(top_entities) > 0:
        print(f"\n Top {category} entities:")
        for idx, row in top_entities.iterrows():
            print(f"  {row['entity']}: {row['count']:,} mentions, {row['num_articles']} articles")
        
        # Save to CSV
        top_entities.to_csv(OUTPUTS_DIR / f'top_{category.lower()}_entities.csv', index=False)

# Visualize top organizations
top_orgs = entity_df[entity_df['primary_type'] == 'ORG'].head(30)

fig = go.Figure(data=[
    go.Bar(
        y=top_orgs['entity'][::-1],
        x=top_orgs['count'][::-1],
        orientation='h',
        marker=dict(color=top_orgs['count'][::-1], colorscale='Blues')
    )
])

fig.update_layout(
    title="Top 30 Organizations Mentioned in Financial News",
    xaxis_title="Number of Mentions",
    yaxis_title="Organization",
    height=800
)

fig.write_html(VIZ_DIR / 'top_organizations.html')
fig.show()


 Top ORG entities:
  Yahoo Finance: 12 mentions, 12 articles
  Boeing: 7 mentions, 4 articles
  Abbott Laboratories: 7 mentions, 4 articles
  Bank of America: 6 mentions, 3 articles
  BA: 6 mentions, 4 articles
  ABT: 5 mentions, 4 articles
  AI: 5 mentions, 4 articles
  APA: 5 mentions, 3 articles
  NYSE: 5 mentions, 5 articles
  American Tower: 4 mentions, 2 articles
  In: 4 mentions, 3 articles
  AvalonBay Communities: 4 mentions, 2 articles
  BABA: 4 mentions, 4 articles
  Broadcom: 3 mentions, 3 articles
  AMD: 3 mentions, 2 articles
  Goldman Sachs: 3 mentions, 2 articles
  Wells Fargo: 3 mentions, 2 articles
  Alibaba Group Holding Limited: 3 mentions, 2 articles
  Alibaba: 3 mentions, 2 articles
  Amazon: 3 mentions, 2 articles

 Top PERSON entities:
  Banco Bradesco: 2 mentions, 1 articles
  Hock Tan: 1 mentions, 1 articles
  Share Price: 1 mentions, 1 articles
  BABA Shares Sold: 1 mentions, 1 articles
  Lobbying Update: 1 mentions, 1 articles
  AMGN: 1 mentions, 1 articles


## 10. Entity Co-Mention Network

In [12]:
# Build co-mention network (companies mentioned together)
print(" Building entity co-mention network...")

# Get top 50 organizations for network analysis
top_entities = entity_df[entity_df['primary_type'] == 'ORG'].head(50)['entity'].tolist()

# Build co-mention matrix
co_mentions = defaultdict(int)

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Computing co-mentions"):
    entities_in_article = [ent['text'] for ent in row['all_entities'] 
                          if ent['text'] in top_entities]
    
    # Create pairs
    for i, ent1 in enumerate(entities_in_article):
        for ent2 in entities_in_article[i+1:]:
            pair = tuple(sorted([ent1, ent2]))
            co_mentions[pair] += 1

# Filter co-mentions (minimum 5 co-occurrences)
min_co_mentions = 5
filtered_co_mentions = {k: v for k, v in co_mentions.items() if v >= min_co_mentions}

print(f"\n Found {len(filtered_co_mentions):,} entity pairs with {min_co_mentions}+ co-mentions")

# Create network graph
G = nx.Graph()

for (ent1, ent2), weight in filtered_co_mentions.items():
    G.add_edge(ent1, ent2, weight=weight)

print(f" Network statistics:")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Density: {nx.density(G):.4f}")

# Calculate centrality metrics
degree_centrality = nx.degree_centrality(G)
betweenness_centrality = nx.betweenness_centrality(G)

# Top entities by centrality
print(f"\n Top entities by network centrality:")
for entity, score in sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {entity}: {score:.3f}")

# Save network data
network_df = pd.DataFrame([
    {'entity1': e1, 'entity2': e2, 'co_mentions': w}
    for (e1, e2), w in filtered_co_mentions.items()
]).sort_values('co_mentions', ascending=False)

network_df.to_csv(OUTPUTS_DIR / 'entity_co_mention_network.csv', index=False)
print(f"\n Saved co-mention network to {OUTPUTS_DIR / 'entity_co_mention_network.csv'}")

 Building entity co-mention network...


Computing co-mentions:   0%|          | 0/63 [00:00<?, ?it/s]


 Found 1 entity pairs with 5+ co-mentions
 Network statistics:
  Nodes: 2
  Edges: 1
  Density: 1.0000

 Top entities by network centrality:
  BA: 1.000
  Boeing: 1.000

 Saved co-mention network to c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\entity_co_mention_network.csv


## 11. Visualize Entity Network

In [13]:
# Visualize network with plotly
import plotly.graph_objects as go

# Use spring layout
pos = nx.spring_layout(G, k=0.5, iterations=50)

# Create edges
edge_x = []
edge_y = []
edge_weights = []

for edge in G.edges(data=True):
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])
    edge_weights.append(edge[2]['weight'])

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=0.5, color='#888'),
    hoverinfo='none',
    mode='lines'
)

# Create nodes
node_x = []
node_y = []
node_text = []
node_size = []

for node in G.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)
    
    # Node size based on degree
    degree = G.degree(node)
    node_size.append(10 + degree * 2)
    
    # Node text
    node_text.append(f"{node}<br>Connections: {degree}")

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers+text',
    text=[node for node in G.nodes()],
    textposition='top center',
    hoverinfo='text',
    hovertext=node_text,
    marker=dict(
        size=node_size,
        color=list(degree_centrality.values()),
        colorscale='YlOrRd',
        colorbar=dict(
            title="Centrality",
            xanchor='left',
            titleside='right'
        ),
        line=dict(width=2, color='white')
    )
)

# Create figure
fig = go.Figure(data=[edge_trace, node_trace],
             layout=go.Layout(
                title='Entity Co-Mention Network<br>(Companies Mentioned Together in Articles)',
                titlefont=dict(size=16),
                showlegend=False,
                hovermode='closest',
                margin=dict(b=20,l=5,r=5,t=40),
                xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                height=800
                ))

fig.write_html(VIZ_DIR / 'entity_network.html')
fig.show()

print("\n Network visualization saved")


 Network visualization saved


## 12. Save Enhanced Dataset

In [ ]:
# Helper function to convert numpy types to Python types for JSON serialization
def convert_entity_to_json_serializable(entity):
    """Convert entity dict with numpy types to JSON-serializable format"""
    json_entity = {}
    for key, value in entity.items():
        if isinstance(value, (np.float32, np.float64)):
            json_entity[key] = float(value)
        elif isinstance(value, (np.int32, np.int64)):
            json_entity[key] = int(value)
        else:
            json_entity[key] = value
    return json_entity

# Convert entity lists to JSON strings for CSV storage
df['spacy_entities_json'] = df['spacy_entities'].apply(
    lambda entities: json.dumps([convert_entity_to_json_serializable(e) for e in entities])
)
df['bert_entities_json'] = df['bert_entities'].apply(
    lambda entities: json.dumps([convert_entity_to_json_serializable(e) for e in entities])
)
df['all_entities_json'] = df['all_entities'].apply(
    lambda entities: json.dumps([convert_entity_to_json_serializable(e) for e in entities])
)

# Extract entity counts by type for each article
df['num_entities'] = df['all_entities'].apply(len)
df['num_orgs'] = df['all_entities'].apply(lambda x: sum(1 for e in x if e['label'] in ['ORG', 'ORGANIZATION']))
df['num_persons'] = df['all_entities'].apply(lambda x: sum(1 for e in x if e['label'] in ['PERSON', 'PER']))
df['num_locations'] = df['all_entities'].apply(lambda x: sum(1 for e in x if e['label'] in ['GPE', 'LOC', 'LOCATION']))

# Save to CSV (drop the original list columns)
df_to_save = df.drop(columns=['spacy_entities', 'bert_entities', 'all_entities'])
df_to_save.to_csv(FEATURES_DIR / 'dataset_with_entities.csv', index=False)

print(f"\n Saved enhanced dataset to {FEATURES_DIR / 'dataset_with_entities.csv'}")
print(f"Dataset shape: {df_to_save.shape}")
print(f"\n New entity columns added:")

df_to_save.head()


 Saved enhanced dataset to c:\Users\HARSHIT\Desktop\p\nlp analysis\01_DATA\features\dataset_with_entities.csv
Dataset shape: (63, 26)

 New entity columns added:
  - spacy_entities_json, bert_entities_json, all_entities_json
  - num_entities, num_orgs, num_persons, num_locations


,date,title,text,source,url,ticker,publisher,query,article_length,word_count,...,textblob_polarity,textblob_label,date_only,spacy_entities_json,bert_entities_json,all_entities_json,num_entities,num_orgs,num_persons,num_locations
0,2026-01-18 12:19:48,American Express Company $AXP Shares Acquired ...,American Express Company $AXP Shares Acquired ...,Google News,https://news.google.com/rss/articles/CBMi2wFBV...,AXP,NaN,AXP stock news,106,13,...,0.000000,neutral,2026-01-18,"[{""text"": ""American Express Company $AXP Share...","[{""text"": ""American Express Company"", ""label"":...","[{""text"": ""American Express Company $AXP Share...",7,6,0,1
1,2026-01-18 12:15:12,Apple stock price slips to $255 ahead of a hol...,Apple stock price slips to $255 ahead of a hol...,Google News,https://news.google.com/rss/articles/CBMiuAFBV...,AAPL,NaN,AAPL stock news,121,18,...,0.000000,neutral,2026-01-18,"[{""text"": ""Apple"", ""label"": ""ORG"", ""start"": 0,...","[{""text"": ""Apple"", ""label"": ""ORG"", ""score"": 0....","[{""text"": ""Apple"", ""label"": ""ORG"", ""source"": ""...",5,4,0,0
2,2026-01-18 12:00:26,Amgen Inc. (NASDAQ:AMGN) is largely controlled...,Amgen Inc. (NASDAQ:AMGN) is largely controlled...,Google News,https://news.google.com/rss/articles/CBMigAFBV...,AMGN,NaN,AMGN stock news,128,16,...,0.407143,positive,2026-01-18,"[{""text"": ""Amgen Inc."", ""label"": ""ORG"", ""start...","[{""text"": ""Amgen Inc"", ""label"": ""ORG"", ""score""...","[{""text"": ""Amgen Inc."", ""label"": ""ORG"", ""sourc...",7,5,1,0
3,2026-01-18 11:43:58,BAC Stock Gains On Bullish Q4 Print Guides For...,BAC Stock Gains On Bullish Q4 Print Guides For...,Google News,https://news.google.com/rss/articles/CBMisgFBV...,BAC,NaN,BAC stock news,100,15,...,0.000000,neutral,2026-01-18,"[{""text"": ""BAC Stock Gains On Bullish Q4 Print...",[],"[{""text"": ""BAC Stock Gains On Bullish Q4 Print...",2,1,0,1
4,2026-01-18 11:26:14,Strong Analyst Confidence in Alibaba Group Hol...,Strong Analyst Confidence in Alibaba Group Hol...,Google News,https://news.google.com/rss/articles/CBMiswFBV...,BABA,NaN,BABA stock news,102,13,...,0.433333,positive,2026-01-18,"[{""text"": ""Alibaba Group Holding"", ""label"": ""O...","[{""text"": ""Alibaba Group Holding"", ""label"": ""O...","[{""text"": ""Alibaba Group Holding"", ""label"": ""O...",5,5,0,0


## 13. Summary Statistics

In [17]:
# Create summary report
summary = {
    'total_articles': len(df),
    'total_entities_extracted': sum(df['num_entities']),
    'unique_entities': len(entity_df),
    'avg_entities_per_article': df['num_entities'].mean(),
    'median_entities_per_article': df['num_entities'].median(),
    'entity_types': dict(entity_type_counts),
    'top_10_entities': entity_df.head(10)[['entity', 'count', 'primary_type']].to_dict('records'),
    'network_nodes': G.number_of_nodes(),
    'network_edges': G.number_of_edges(),
    'network_density': nx.density(G),
    'most_central_entities': sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:10]
}

# Save summary
with open(OUTPUTS_DIR / 'ner_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(" Named Entity Recognition Summary")
print("=" * 50)
print(f"Total articles processed: {summary['total_articles']:,}")
print(f"Total entities extracted: {summary['total_entities_extracted']:,}")
print(f"Unique entities: {summary['unique_entities']:,}")
print(f"Average entities per article: {summary['avg_entities_per_article']:.2f}")
print(f"Median entities per article: {summary['median_entities_per_article']:.0f}")
print(f"\nNetwork statistics:")
print(f"  Nodes: {summary['network_nodes']}")
print(f"  Edges: {summary['network_edges']}")
print(f"  Density: {summary['network_density']:.4f}")

print(f"\n Summary saved to {OUTPUTS_DIR / 'ner_summary.json'}")
print("\n Named Entity Recognition complete!")
print("\n Output files:")
print(f"  - {FEATURES_DIR / 'dataset_with_entities.csv'}")
print(f"  - {OUTPUTS_DIR / 'financial_entity_database.csv'}")
print(f"  - {OUTPUTS_DIR / 'entity_co_mention_network.csv'}")
print(f"  - {VIZ_DIR / 'entity_type_distribution.html'}")
print(f"  - {VIZ_DIR / 'top_organizations.html'}")
print(f"  - {VIZ_DIR / 'entity_network.html'}")

 Named Entity Recognition Summary
Total articles processed: 63
Total entities extracted: 374
Unique entities: 226
Average entities per article: 5.94
Median entities per article: 6

Network statistics:
  Nodes: 2
  Edges: 1
  Density: 1.0000

 Summary saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\ner_summary.json

 Named Entity Recognition complete!

 Output files:
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\01_DATA\features\dataset_with_entities.csv
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\financial_entity_database.csv
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\entity_co_mention_network.csv
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\visualizations\entity_type_distribution.html
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\visualizations\top_organizations.html
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\visualizations\entity_network.html
